# Cross-Sectional Model Comparison: bayespecon vs spreg

This notebook compares equivalent cross-sectional models between `bayespecon` and `spreg` on a real example dataset from the PySAL/spreg ecosystem.

Model mapping used:

- `SLX` (bayespecon) vs `spreg.OLS(slx_lags=1)`
- `SAR` (bayespecon) vs `spreg.GM_Lag`
- `SEM` (bayespecon) vs `spreg.GM_Error`
- `SDM` (bayespecon) vs `spreg.GM_Lag(slx_lags=1, w_lags=2)`
- `SDEM` (bayespecon) vs `spreg.GM_Error(slx_lags=1)`

Dataset: **Columbus** neighborhood data from `libpysal.examples`.

In [9]:
import numpy as np
import pandas as pd
import libpysal
import spreg
from libpysal.weights import Queen
from libpysal.graph import Graph

from IPython.display import display
from bayespecon import SLX, SAR, SEM, SDM, SDEM

In [10]:
# Load a real dataset from libpysal/spreg examples
columbus = libpysal.examples.load_example("Columbus")
shp_path = columbus.get_path("columbus.shp")
dbf_path = columbus.get_path("columbus.dbf")

db = libpysal.io.open(dbf_path)
y_col = np.asarray(db.by_col("CRIME"), dtype=float)
X_raw = pd.DataFrame({"INC": np.asarray(db.by_col("INC"), dtype=float), "HOVAL": np.asarray(db.by_col("HOVAL"), dtype=float)})

X_spreg = X_raw.to_numpy()
y_spreg = y_col.reshape((-1, 1))
X_bayes = pd.concat([pd.Series(1.0, index=X_raw.index, name="CONSTANT"), X_raw], axis=1)
y_bayes = y_col

w = Queen.from_shapefile(shp_path)
w.transform = "r"
w_coo = w.sparse.tocoo()
W_graph = Graph.from_arrays(w_coo.row, w_coo.col, w_coo.data.astype(float))

print(f"N={len(y_col)} observations")
X_bayes.head()

N=49 observations


/Users/knaaptime/miniforge3/envs/bayespecon/lib/python3.14/site-packages/libpysal/io/iohandlers/pyShpIO.py:247: FutureWarning: Objects based on the `Geometry` class will deprecated and removed in a future version of libpysal.
  shp = self.type(vertices)
/Users/knaaptime/miniforge3/envs/bayespecon/lib/python3.14/site-packages/libpysal/cg/shapes.py:1408: FutureWarning: Objects based on the `Geometry` class will deprecated and removed in a future version of libpysal.
  self._part_rings = [Ring(vertices)]


,CONSTANT,INC,HOVAL
0,1.0,19.531,80.467003
1,1.0,21.232,44.567001
2,1.0,15.956,26.350000
3,1.0,4.477,33.200001
4,1.0,11.252,23.225000


In [11]:
spreg_results = {
    "SLX": spreg.OLS(y_spreg, X_spreg, w=w, slx_lags=1, name_y="CRIME", name_x=["INC", "HOVAL"]),
    "SAR": spreg.GM_Lag(y_spreg, X_spreg, w=w, name_y="CRIME", name_x=["INC", "HOVAL"]),
    "SEM": spreg.GM_Error(y_spreg, X_spreg, w=w, name_y="CRIME", name_x=["INC", "HOVAL"]),
    "SDM": spreg.GM_Lag(y_spreg, X_spreg, w=w, slx_lags=1, w_lags=2, name_y="CRIME", name_x=["INC", "HOVAL"]),
    "SDEM": spreg.GM_Error(y_spreg, X_spreg, w=w, slx_lags=1, name_y="CRIME", name_x=["INC", "HOVAL"]),
}

sample_kwargs = dict(draws=600, tune=600, chains=4, random_seed=42, progressbar=False)
bayes_models = {
    "SLX": SLX(y=y_bayes, X=X_bayes, W=W_graph),
    "SAR": SAR(y=y_bayes, X=X_bayes, W=W_graph),
    "SEM": SEM(y=y_bayes, X=X_bayes, W=W_graph),
    "SDM": SDM(y=y_bayes, X=X_bayes, W=W_graph),
    "SDEM": SDEM(y=y_bayes, X=X_bayes, W=W_graph),
}
bayes_idata = {name: model.fit(**sample_kwargs) for name, model in bayes_models.items()}
print("Finished fitting all bayespecon models.")

Initializing NUTS using jitter+adapt_diag...


GM_Lag
GM_Error
GM_Lag
GM_Error


Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta, sigma]
Sampling 4 chains for 600 tune and 600 draw iterations (2_400 + 2_400 draws total) took 1 seconds.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [rho, beta, sigma]
Sampling 4 chains for 600 tune and 600 draw iterations (2_400 + 2_400 draws total) took 1 seconds.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [lam, beta, sigma]
Sampling 4 chains for 600 tune and 600 draw iterations (2_400 + 2_400 draws total) took 1 seconds.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [rho, beta, sigma]
Sampling 4 chains for 600 tune and 600 draw iterations (2_400 + 2_400 draws total) took 2 seconds.
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [lam, beta, sigma]
Sampling 4 chains for 600 tune and 600 draw iterations (2_400 + 2_400 draws total) took 1

Finished fitting all bayespecon models.


In [12]:
def extract_spreg_params(model):
    names = list(getattr(model, "name_x", []))
    betas = np.asarray(model.betas).flatten()
    if len(betas) == len(names) + 1 and type(model).__name__ == "GM_Lag":
        names = names + ["rho"]
    if len(names) != len(betas):
        names = [f"param_{i}" for i in range(len(betas))]
    return pd.Series(betas, index=names, dtype=float)

def extract_bayespecon_params(model_name, model, idata):
    out = {}
    beta_mean = idata.posterior["beta"].mean(("chain", "draw")).values
    beta_names = model._beta_names() if model_name in {"SLX", "SDM", "SDEM"} else list(model._feature_names)
    for name, val in zip(beta_names, beta_mean):
        out[name] = float(val)
    if model_name in {"SAR", "SDM"}:
        out["rho"] = float(idata.posterior["rho"].mean())
    if model_name in {"SEM", "SDEM"}:
        out["lambda"] = float(idata.posterior["lam"].mean())
    harmonized = {}
    for k, v in out.items():
        key = k.replace("W*", "W_")
        if key.lower() == "intercept":
            key = "CONSTANT"
        harmonized[key] = v
    return pd.Series(harmonized, dtype=float)

rows = []
for model_name in ["SLX", "SAR", "SEM", "SDM", "SDEM"]:
    sp = extract_spreg_params(spreg_results[model_name])
    by = extract_bayespecon_params(model_name, bayes_models[model_name], bayes_idata[model_name])
    common = sorted(set(sp.index).intersection(set(by.index)))
    for param in common:
        rows.append({"model": model_name, "parameter": param, "bayespecon_posterior_mean": by[param], "spreg_estimate": sp[param], "difference": by[param] - sp[param]})

comparison = pd.DataFrame(rows).sort_values(["model", "parameter"]).reset_index(drop=True)


In [13]:
display(comparison)

,model,parameter,bayespecon_posterior_mean,spreg_estimate,difference
0,SAR,CONSTANT,46.177695,43.963191,2.214505
1,SAR,HOVAL,-0.266630,-0.265793,-0.000837
2,SAR,INC,-1.061617,-1.009637,-0.051980
3,SAR,rho,0.413482,0.453491,-0.040009
4,SDEM,CONSTANT,72.986752,73.782584,-0.795832
5,SDEM,HOVAL,-0.285482,-0.278631,-0.006851
6,SDEM,INC,-1.004101,-1.058484,0.054383
7,SDEM,W_HOVAL,0.104295,0.138482,-0.034187
8,SDEM,W_INC,-1.110198,-1.225845,0.115648
9,SDEM,lambda,0.471228,0.371442,0.099786


`bayespecon` uses Bayesian inference, so exact equality with `spreg` point estimates is not expected. The comparison table is intended to confirm that equivalent model specifications produce directionally and numerically comparable coefficients on the same real dataset.